<a href="https://colab.research.google.com/github/vsolanki76771215/Student_MLE_MiniProject_EDA/blob/main/PrototypeScaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 1 — Scalable scikit-learn Prototype
# Uses incremental learning for large tabular data
# ============================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

DATA_PATH = "sample_data/scaled_deforestation_ml_dataset.csv"

feature_cols = [
    "ndvi_2018_mean",
    "ndvi_2022_mean",
    "ndvi_diff",
    "treecover_mean"
]

target_col = "loss_binary"

df = pd.read_csv(DATA_PATH)

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

sgd_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

sgd_pipeline.fit(X_train, y_train)

y_pred = sgd_pipeline.predict(X_test)
y_prob = sgd_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.87      0.89     26939
           1       0.41      0.51      0.46      4892

    accuracy                           0.81     31831
   macro avg       0.66      0.69      0.67     31831
weighted avg       0.83      0.81      0.82     31831

Confusion Matrix:
[[23381  3558]
 [ 2388  2504]]
ROC-AUC: 0.8066392775817033
PR-AUC: 0.35795040552536983


In [ ]:
# ============================================================
# STEP 2 — Scalable Random Forest Baseline
# Useful comparison model for structured satellite features
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("ROC-AUC:", roc_auc_score(y_test, rf_prob))
print("PR-AUC:", average_precision_score(y_test, rf_prob))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.88      0.92     26939
           1       0.55      0.78      0.65      4892

    accuracy                           0.87     31831
   macro avg       0.75      0.83      0.78     31831
weighted avg       0.89      0.87      0.88     31831

Random Forest Confusion Matrix:
[[23826  3113]
 [ 1066  3826]]
ROC-AUC: 0.9263275510824446
PR-AUC: 0.7097944398118861


In [ ]:
# ============================================================
# STEP 3 — TensorFlow / Keras Scalable Deep Learning Prototype
# Uses tf.data batching for larger datasets
# ============================================================

import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

df = pd.read_csv("sample_data/scaled_deforestation_ml_dataset.csv")

X = df[feature_cols].values.astype("float32")
y = df[target_col].values.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

BATCH_SIZE = 4096

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(50_000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            patience=4,
            mode="max",
            restore_best_weights=True
        )
    ]
)

y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.7257 - loss: 0.6601 - pr_auc: 0.3989 - roc_auc: 0.7858 - val_accuracy: 0.8478 - val_loss: 0.5792 - val_pr_auc: 0.4967 - val_roc_auc: 0.7387
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.7931 - loss: 0.5518 - pr_auc: 0.4738 - roc_auc: 0.8156 - val_accuracy: 0.8465 - val_loss: 0.5295 - val_pr_auc: 0.5278 - val_roc_auc: 0.7719
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.8255 - loss: 0.4862 - pr_auc: 0.4987 - roc_auc: 0.8257 - val_accuracy: 0.8466 - val_loss: 0.4789 - val_pr_auc: 0.5519 - val_roc_auc: 0.8260
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.8421 - loss: 0.4332 - pr_auc: 0.5085 - roc_auc: 0.8293 - val_accuracy: 0.8464 - val_loss: 0.4335 - val_pr_auc: 0.5521 - val_roc_auc: 0.8254
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.8542 - loss: 0.3905 - pr_auc: 0.5171 - roc_auc: 0.8370 - val_accuracy: 0.8466 - val_loss: 0.4002 - val_pr_auc: 0.55

Data Location: https://drive.google.com/file/d/1w4DzZAwbuambpbttAiTg3qhfB1ZNRQS8/view?usp=sharing




## **Objective**

The objective of this phase of the capstone project was to demonstrate that the proposed machine learning solution can scale beyond the initial prototype and operate on a substantially larger dataset representative of a real-world environmental monitoring system. The previous phase established that the selected machine learning algorithms were capable of detecting deforestation using Sentinel-2 derived vegetation indices and Hansen Global Forest Change labels. This phase focused on increasing the volume of data processed while maintaining data quality and evaluating whether the algorithms could continue to perform effectively at scale.

Unlike the earlier prototype, which was designed to validate feasibility, the scaled implementation emphasizes efficient data generation, processing, model training, and evaluation on a much larger dataset.



## **Scaling the Dataset**

The original dataset contained approximately 91,000 satellite image patches extracted from three Areas of Interest (AOIs) in the Madre de Dios region of Peru:


*   La Pampa
*   Tambopata
*   Madre de Dios Corridor

Each patch represented a 32×32 pixel region containing vegetation characteristics derived from Sentinel-2 imagery and forest-loss labels obtained from the Hansen Global Forest Change dataset.

Instead of generating synthetic observations, the dataset was expanded using an organic data generation approach. The patch extraction process was reconfigured to generate substantially more real observations from the same satellite imagery.


The following configuration changes were made:

*   Patch Stride:	From 16 pixels	to 8 pixels
*   Target Patches per AOI:	From 20,000	to 70,000


Reducing the stride from 16 pixels to 8 pixels increases the density of the sliding window used during patch extraction. Rather than skipping every 16 pixels, the extraction process samples every 8 pixels, producing overlapping patches and significantly increasing the number of valid observations without introducing artificial or synthetic data.

To maintain dataset quality, the patch generation pipeline continued to apply several filtering criteria:

*   Removal of patches containing excessive invalid pixels
*   Balanced sampling between deforestation and non-deforestation regions
*   Removal of duplicate patches
*   Matching of Sentinel-2 feature patches with Hansen label patches using a common patch index


This approach ensures that every additional observation originates from actual satellite imagery rather than synthetic oversampling techniques.

## **Scaling Decisions and Trade-offs**

Several implementation decisions were required while scaling the prototype.


**1. Organic Data Expansion vs. Synthetic Data Generation**

One possible approach was to generate additional observations using synthetic sampling techniques such as bootstrapping or noise injection.

This approach was intentionally rejected because synthetic samples do not represent new environmental observations and may artificially inflate model performance.

Instead, the project generated additional data directly from the original satellite imagery by increasing patch extraction density. This produces genuine observations while preserving the statistical characteristics of the underlying satellite data.

Decision


*   Not generating synthetic data generation
*   Generating organic expansion using additional overlapping image patches

This approach provides a more realistic evaluation of how the model would perform in production.


**2. Patch Size**

The original patch size of 32 × 32 pixels was retained.

Smaller patches (for example 16 × 16) would have produced many more training examples but would also remove important spatial context surrounding deforestation events.

Larger patches would reduce the total number of observations and increase computational cost.

The selected patch size provided an effective balance between contextual information and computational efficiency.


**3. Patch Stride**

The stride controls the spacing between adjacent image patches.

Original prototype:

Stride = 16 pixels

Scaled prototype:

Stride = 8 pixels

Reducing the stride approximately quadruples the number of candidate patches extracted from each AOI because the sliding window moves more frequently across the raster.

**Advantages:**

*   Significantly more training data
*   Better representation of transition regions
*   Improved coverage of deforestation boundaries


**Trade-off:**
*   Increased storage requirements
*   Longer preprocessing time
*   Higher computational cost during model training

**4. Maintaining Real Labels**

An important design decision was to continue using Hansen Global Forest Change labels for every generated patch.

Each generated patch remained paired with:


*   NDVI 2018
*   NDVI 2022
*   NDVI Difference
*   Tree Cover
*   Binary forest-loss label


This ensured that every observation corresponded to an actual geographic location and verified forest-loss event.

## **Choice of Tools and Libraries**

Several machine learning frameworks were evaluated for scalability.

**scikit-learn**

Scikit-learn served as the primary framework for classical machine learning.

Advantages include:

*   Efficient implementation for structured tabular datasets
*   Parallel processing support
*   Mature model selection and evaluation utilities
*   Easy integration with preprocessing pipelines


The following algorithms were evaluated:
*   Logistic Regression
*   Decision Tree
*   Random Forest
*   Gradient Boosting
*   Extra Trees
*   AdaBoost
*   Ensemble methods

Among these algorithms, Random Forest consistently produced the strongest performance while maintaining reasonable training time.

**TensorFlow / Keras**

TensorFlow and Keras were used to investigate whether a neural network could improve predictive performance.

The deep learning implementation included:

*   Fully connected neural network
*   Batch Normalization
*   Dropout regularization
*   Adam optimizer
*   Early stopping
*   tf.data data pipeline
*   Mini-batch training

The tf.data API allowed the dataset to be streamed efficiently in batches rather than loading the entire dataset into memory, making the implementation suitable for much larger datasets.

Although the neural network achieved high overall accuracy, it underperformed Random Forest on the minority deforestation class.

**SparkML**

SparkML was investigated as the preferred framework for distributed processing.

SparkML becomes advantageous when datasets exceed the memory capacity of a single machine.

Advantages include:

*   Distributed feature engineering
*   Distributed model training
*   Horizontal scalability
*   Integration with cloud computing platforms such as Databricks and EMR

For the current dataset size, distributed computation was unnecessary because the entire dataset comfortably fit within system memory. Nevertheless, SparkML remains the recommended solution for future deployment involving nationwide or continental-scale satellite imagery.

**PyTorch**

PyTorch was considered for future deep learning development.

Its advantages include:

*   Flexible dynamic computation graphs
*   Advanced CNN implementation
*   U-Net semantic segmentation
*   Vision Transformer architectures

Because the current project uses tabular NDVI features rather than raw image tensors, TensorFlow/Keras provided a simpler implementation. PyTorch would become more appropriate when transitioning to semantic segmentation using full Sentinel-2 imagery.

## **Choice of Machine Learning Technique**

The dataset contains structured numerical features:

*   NDVI 2018 Mean
*   NDVI 2022 Mean
*   NDVI Difference
*   Tree Cover Mean

These features are well suited to ensemble tree-based learning methods.

Several candidate models were evaluated.

**Random Forest**

Random Forest demonstrated the strongest overall performance.

Final performance:

*   Accuracy	87%
*   Precision (Deforestation)	55%
*   Recall (Deforestation)	78%
*   F1-score	0.65
*   ROC-AUC	0.926
*   PR-AUC	0.710

Random Forest effectively captured nonlinear relationships among vegetation indices while remaining robust to feature scaling and outliers. Most importantly, it achieved a high recall for deforestation detection, making it suitable for environmental monitoring where missing illegal activity is more costly than investigating false positives.

**TensorFlow/Keras Neural Network**

The neural network achieved:

*   Accuracy	88%
*   ROC-AUC	0.892
*   PR-AUC	0.620

While overall accuracy was slightly higher, the model detected substantially fewer deforestation patches.

Recall decreased from:

Random Forest: 78%

to

Neural Network: 33%

This conservative behavior resulted in fewer false alarms but many more missed deforestation events.

For environmental monitoring, recall is more important than overall accuracy because failing to detect illegal deforestation has greater consequences than conducting additional investigations.

## **Final Model Selection**

Although both models demonstrated good performance on the scaled dataset, the Random Forest classifier was selected as the final production model.

The decision was based on several factors:

*   Highest ROC-AUC (0.926)
*   Highest PR-AUC (0.710)
*   Significantly better recall for the minority class
*   Better F1-score
*   Faster training time
*   Greater interpretability
*   Excellent performance on structured tabular satellite-derived features

The TensorFlow neural network serves as a valuable benchmark and demonstrates the feasibility of deep learning for this problem. However, given the current feature representation, the ensemble-based Random Forest model provides superior detection performance and better satisfies the objectives of identifying deforestation events.

## **Scalability Assessment**


The scaled implementation demonstrates that the proposed machine learning pipeline can efficiently process a substantially larger volume of satellite-derived observations without requiring fundamental changes to the underlying workflow. By increasing patch extraction density and optimizing preprocessing, the dataset was expanded using only real Sentinel-2 imagery and Hansen Global Forest Change labels, preserving data integrity while significantly increasing sample size. The preprocessing pipeline, feature extraction, and model training stages remained stable under the increased workload.


From a deployment perspective, the selected architecture can be further scaled by incorporating additional Areas of Interest, processing longer temporal sequences, or migrating preprocessing and model training to distributed frameworks such as SparkML. While TensorFlow/Keras provides a pathway toward future deep learning approaches using full satellite image patches, the Random Forest classifier proved to be the most effective solution for the current structured feature set, offering the best balance of predictive performance, computational efficiency, scalability, and interpretability. These results demonstrate that the proposed solution is technically viable for operational large-scale monitoring of deforestation using satellite imagery.